In [1]:
print(f"HEllo there")

HEllo there


In [6]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root, one level above tests/

from dossier_agent.verification import fetch_page_text, USER_AGENT

/Users/oles/UCU/3_semester/GenAI/Ass1/CHI_GEAR_Agent/.venv/lib/python3.12/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.SKILL_TOOLSET is enabled.
  check_feature_enabled()


In [9]:
import httpx, re
from bs4 import BeautifulSoup

link = "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQH1FdUNl2HC-wrCI011rGzHJYMM0ZYtFhaMqJoJ3AXcfUWxdTOuRdglplygp3Osgg8xgp3RRIOH_UG5eBjuEfUZSkfsM5xfdfrlBm6MDtFRAVkq61rxjiG4tSAtfAhVW2Ue3N4="
headers = {"User-Agent": "ProspectDossierBot/0.1 (student project; your-email@example.com)"}
r = httpx.get(link, headers=headers, follow_redirects=True, timeout=15)

text = BeautifulSoup(r.text, "html.parser").get_text(" ")
text = re.sub(r"\s+", " ", text).lower()

for needle in ["deadpool", "founded in 2013", "james proud", "san francisco"]:
    i = text.find(needle)
    print(needle, "->", text[max(0, i-80): i+120] if i >= 0 else "NOT FOUND")

deadpool -> NOT FOUND
founded in 2013 -> NOT FOUND
james proud ->  2012 dissolved june 2017 headquarters san francisco , california, us [ 1 ] ceo james proud website hello.is at the wayback machine (archived january 28, 2017) native clients on ios , android [ 2 ] he
san francisco -> ology , sleep tracking founded august 28, 2012 dissolved june 2017 headquarters san francisco , california, us [ 1 ] ceo james proud website hello.is at the wayback machine (archived january 28, 2017)


In [10]:
import re

def tokens(s: str) -> list[str]:
    s = re.sub(r"\[\s*\d+\s*\]", " ", s)       # drop footnote markers like [ 1 ]
    return re.findall(r"[a-z0-9]+", s.lower())  # words and numbers only

quote = "Hello was an American technology company that sold sleep tracking devices and a sleep tracking application to help monitor sleep."
q, p = tokens(quote), tokens(text)               # text from the previous snippet

def contains_seq(page, q):
    n = len(q)
    return any(page[i:i+n] == q for i in range(len(page) - n + 1))

print("exact tokens match:", contains_seq(p, q))
print("share of quote words on page:", sum(w in set(p) for w in q) / len(q))

exact tokens match: True
share of quote words on page: 1.0


In [8]:
import asyncio, httpx
from dossier_agent.verification import fetch_page_text, USER_AGENT

URLS = [
    "https://plausible.io/enterprise",
    "https://plausible.io/ai-traffic-analytics",
    "https://plausible.io/google-analytics-pricing",
    "https://plausible.io/direct-traffic",
    "https://plausible.io/web-analytics-vs-product-analytics",
    "https://plausible.io/public-sector",
    "https://plausible.io/docs/team",
]

async def main():
    async with httpx.AsyncClient(headers={"User-Agent": USER_AGENT}, timeout=15) as client:
        for url in URLS:
            r = await fetch_page_text(url, client)
            print(r.ok, r.error, r.final_url, url)

await main()

False HTTP 404 https://plausible.io/enterprise https://plausible.io/enterprise
False HTTP 404 https://plausible.io/ai-traffic-analytics https://plausible.io/ai-traffic-analytics
False HTTP 404 https://plausible.io/google-analytics-pricing https://plausible.io/google-analytics-pricing
False HTTP 404 https://plausible.io/direct-traffic https://plausible.io/direct-traffic
False HTTP 404 https://plausible.io/web-analytics-vs-product-analytics https://plausible.io/web-analytics-vs-product-analytics
False HTTP 404 https://plausible.io/public-sector https://plausible.io/public-sector
False HTTP 404 https://plausible.io/docs/team https://plausible.io/docs/team
